In [8]:
import pandas as pd
import numpy as np
from scipy import stats

In [10]:
# Load our standardized dataset
df = pd.read_csv("../data/processed/marketing_cleaned.csv")

In [11]:
#--- SAMPLE RATIO MISMATCH (SRM) TEST ---
total_obs = len(df)
observed_counts = df['test_group'].value_counts()
n_ad_obs = observed_counts['ad']
n_psa_obs = observed_counts['psa']

# Expected split is 96% ad and 4% psa
expected_p_ad = 0.96
expected_p_psa = 0.04
expected_counts = [total_obs * expected_p_ad, total_obs * expected_p_psa]
observed = [n_ad_obs, n_psa_obs]

chi2_stat, srm_p_value = stats.chisquare(f_obs=observed, f_exp=expected_counts)

print("--- SAMPLE RATIO MISMATCH (SRM) AUDIT ---")
print(f"Total Observations: {total_obs:,}")
print(f"Observed - Ad: {n_ad_obs:,} ({n_ad_obs/total_obs*100:.4f}%), PSA: {n_psa_obs:,} ({n_psa_obs/total_obs*100:.4f}%)")
print(f"Expected - Ad: {expected_counts[0]:,.1f} (96.0%), PSA: {expected_counts[1]:,.1f} (4.0%)")
print(f"Chi-square Statistic: {chi2_stat:.5f}")
print(f"p-value: {srm_p_value:.5f}")

if srm_p_value < 0.01:
    print("WARNING: Statistically significant SRM detected at alpha = 0.01!")
else:
    print("RESULT: No evidence of SRM. Observed split matches the 96:4 design.")

--- SAMPLE RATIO MISMATCH (SRM) AUDIT ---
Total Observations: 588,101
Observed - Ad: 564,577 (96.0000%), PSA: 23,524 (4.0000%)
Expected - Ad: 564,577.0 (96.0%), PSA: 23,524.0 (4.0%)
Chi-square Statistic: 0.00000
p-value: 0.99979
RESULT: No evidence of SRM. Observed split matches the 96:4 design.


In [12]:
# --- COVARIATE & EXPOSURE BALANCE CHECKS ---
print("\n--- EXPOSURE COMPARISON: total_ads ---")
exposure_summary = df.groupby('test_group')['total_ads'].agg(
    mean='mean',
    median='median',
    std='std',
    iqr=lambda x: np.percentile(x, 75) - np.percentile(x, 25)
)
print(exposure_summary)

# Day of week distribution comparison
print("\n--- DAY OF WEEK PROPORTION BY GROUP (%) ---")
day_crosstab = pd.crosstab(df['most_ads_day'], df['test_group'], normalize='columns') * 100
day_crosstab['diff_pct_points'] = day_crosstab['ad'] - day_crosstab['psa']
print(day_crosstab.round(2))


--- EXPOSURE COMPARISON: total_ads ---
                 mean  median        std   iqr
test_group                                    
ad          24.823365    13.0  43.750456  23.0
psa         24.761138    12.0  42.860720  22.0

--- DAY OF WEEK PROPORTION BY GROUP (%) ---
test_group       ad    psa  diff_pct_points
most_ads_day                               
Friday        15.73  16.17            -0.44
Monday        14.80  14.89            -0.08
Saturday      13.96  12.15             1.81
Sunday        14.58  13.00             1.58
Thursday      14.01  16.60            -2.59
Tuesday       13.21  12.36             0.85
Wednesday     13.71  14.84            -1.12
